# M4 · Limpieza de datos

**Bootcamp de Datos para Funcionarios Públicos — Formación Pública**
Módulo compartido · Tronco común (Semana 5) · Se ejecuta en Google Colab.

## ¿Qué vas a lograr?
En **M3** exploraste una tabla que ya venía ordenada. Pero los datos reales del Estado **casi nunca llegan limpios**: tienen espacios de más, mayúsculas inconsistentes, montos guardados como texto, celdas vacías y filas repetidas. Antes de analizar cualquier cosa, hay que **dejar la tabla en condiciones**. Eso es la limpieza de datos, y es donde un analista pasa buena parte de su tiempo.

**Competencia de salida:** diagnosticar los problemas de un dataset sucio y dejarlo listo para analizar — texto normalizado, columnas numéricas convertidas, sin faltantes ni duplicados.

**Entregable:** que las cinco celdas de chequeo muestren ✅.


## Los datos que vamos a usar
Volvemos al **caso conductor del bootcamp: las compras públicas (ChileCompra)**. Usaremos una cifra real y actual: los **rubros más comprados por el Estado** durante 2026, con su monto transado bruto.

| Rubro | Monto bruto 2026 |
| --- | --- |
| Organizaciones y consultorías de administración pública | $713.645.989.343 |
| Medicamentos y productos farmacéuticos | $584.975.029.638 |
| Servicios de construcción y mantenimiento | $500.625.771.226 |
| Equipamiento y suministros médicos | $432.209.437.744 |
| Tecnologías de la información | $190.800.005.495 |
| Servicios de defensa nacional y orden | $181.577.319.021 |

> **Fuente:** Dirección ChileCompra — *Datos Abiertos* (datos-abiertos.chilecompra.cl), año 2026. En 2026, **1.171 organismos** generaron **555.512 órdenes de compra**. Los montos son brutos (neto + impuestos).

### ¿Por qué estos datos vienen "sucios"?
No es descuido: es la naturaleza del dato. El propio portal de ChileCompra advierte que la información se publica **"tal como fue ingresada por los usuarios compradores y proveedores"**, es decir, son **datos autodeclarados** por miles de funcionarios distintos. Cuando mucha gente escribe a mano, aparecen espacios de más, un mismo rubro escrito en MAYÚSCULAS y en minúsculas, montos con símbolo de peso, celdas que quedaron vacías y filas cargadas dos veces. **Tu trabajo es detectar y corregir todo eso.**

Para practicar, en este módulo cargamos un *export crudo* de esa tabla con los defectos típicos que tendría en la vida real.


## Preparar los datos
Ejecuta la celda de abajo para preparar el entorno. Carga el archivo `compras_rubros.csv` (el *export crudo*) y define una función `cargar()` que devuelve una copia fresca de la tabla. Usaremos `cargar()` cada vez que empecemos un ejercicio, para que ninguno dependa de los anteriores.

*(Nota: Para este módulo, el archivo `compras_rubros.csv` ya viene guardado de forma fija y estática en la carpeta de tu proyecto. Fuente de los datos: Dirección ChileCompra — Datos Abiertos, año 2026)*

In [ ]:
import os
import urllib.request
import pandas as pd

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("compras_rubros.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/A3-limpieza-de-datos/compras_rubros.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "compras_rubros.csv")
        print("compras_rubros.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar compras_rubros.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

def cargar():
    """Devuelve una copia limpia-de-estado del export crudo (siempre igual)."""
    return pd.read_csv("compras_rubros.csv")

df = cargar()
print("Filas:", len(df))
df

## 1. La regla de oro: diagnosticar antes de limpiar
Imagina que llegas a una **bodega desordenada**. Antes de ponerte a mover cajas, primero das una vuelta para ver *qué* está mal: cajas sin etiqueta, dos cajas iguales, una etiqueta borrosa. Con los datos es igual: **primero se diagnostica, después se arregla.**

pandas tiene cuatro herramientas de diagnóstico que conviene correr siempre al abrir una tabla nueva:

- `df.info()` → tipos de cada columna y cuántos valores no nulos tiene cada una.
- `df.isnull().sum()` → cuántas **celdas vacías** (faltantes) hay por columna.
- `df.duplicated().sum()` → cuántas **filas repetidas** hay.
- `df["columna"].unique()` → los **valores distintos** de una columna (sirve para detectar el mismo dato escrito de varias formas).


In [ ]:
df = cargar()

# Tipos y conteo de no-nulos por columna
df.info()

In [ ]:
# Faltantes por columna, filas duplicadas y valores únicos del rubro
print("Faltantes por columna:")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("\nRubros distintos (ojo a los repetidos 'disfrazados'):")
for r in df["rubro"].unique():
    print(repr(r))   # repr() muestra los espacios invisibles entre comillas

**Fíjate en lo que revela el diagnóstico:**
- La columna `monto` es de tipo *object* (texto), no número: no podríamos sumarla todavía.
- Hay **1 celda vacía** en `monto` (un rubro al que le faltó el dato).
- Hay **1 fila duplicada** (un rubro cargado dos veces).
- En `df["rubro"].unique()`, `repr()` deja ver los **espacios invisibles** (`'  Medicamentos... '`) y las **mayúsculas inconsistentes** (`'EQUIPAMIENTO Y SUMINISTROS MÉDICOS'`). Para una persona "Medicamentos" y " Medicamentos " son lo mismo; para el computador, **no**.


## 2. Limpiar texto: espacios y mayúsculas
Los dos problemas de texto más comunes son los **espacios sobrantes** al inicio o final, y las **mayúsculas inconsistentes**. Son traicioneros porque a simple vista no se ven, pero rompen los conteos: `value_counts()` contaría `"Medicamentos"` y `" Medicamentos"` como dos rubros distintos.

pandas trae un conjunto de operaciones de texto que se aplican a toda una columna con el accesor `.str`:

- `.str.strip()` → quita los espacios del inicio y del final.
- `.str.lower()` / `.str.upper()` → pasa todo a minúsculas / mayúsculas.
- `.str.capitalize()` → primera letra mayúscula y el resto en minúscula (ideal para nombres y rubros en español).
- `.str.replace("a", "b")` → reemplaza un texto por otro.

Se pueden **encadenar**: primero `strip`, luego `capitalize`.


In [ ]:
df = cargar()

# Encadenamos: quitar espacios y normalizar mayúsculas
df["rubro"] = df["rubro"].str.strip().str.capitalize()

for r in df["rubro"].unique():
    print(repr(r))

Ahora `'  Medicamentos y productos farmacéuticos '` y `'EQUIPAMIENTO Y SUMINISTROS MÉDICOS'` quedaron normalizados y comparables. **Importante:** normalizar el texto a veces *revela* duplicados que antes parecían distintos (el mismo rubro escrito de dos formas). Por eso conviene limpiar el texto **antes** de buscar duplicados.


## 3. De texto a número
La columna `monto` llegó como texto: `"$584.975.029.638"`. Mientras sea texto **no se puede sumar ni promediar** (`+` sobre texto pega cadenas, no suma). Hay que convertirla a número, y para eso primero quitamos lo que estorba:

- el símbolo de peso `$`,
- los puntos `.` que en Chile separan los miles.

Quitamos esos caracteres con `.replace(...)` y luego convertimos con `int(...)` (número entero). Para una columna entera, se usa `.astype(int)` o, más seguro, `pd.to_numeric(...)`.

> ⚠️ **Cuidado con los separadores.** En Chile el punto separa miles y la coma los decimales (`$1.234,50`). En inglés es al revés. Si conviertes sin quitar los puntos, `int("584.975")` da error; y si dejas que pandas interprete el punto como decimal, **cambia el valor**. Siempre revisa qué significan el punto y la coma en *tus* datos.


In [ ]:
# Convertir un monto de texto a número entero, paso a paso
texto = "$584.975.029.638"
sin_simbolo = texto.replace("$", "")      # "584.975.029.638"
sin_puntos  = sin_simbolo.replace(".", "") # "584975029638"
numero      = int(sin_puntos.strip())      # 584975029638  (ya es un número)

print(sin_simbolo)
print(sin_puntos)
print(numero, "->", type(numero))
print("El doble es:", numero * 2)          # ahora sí podemos hacer aritmética

## 4. Valores faltantes (NaN)
Una celda vacía se carga en pandas como `NaN` (*Not a Number*, "no es un número"). No es un cero ni un texto vacío: es la marca de **"aquí falta el dato"**. Se detectan con `.isnull()` y tienes dos caminos para resolverlos:

- **Eliminar** las filas incompletas con `.dropna()`. Útil cuando el dato faltante no se puede recuperar y son pocas filas.
- **Rellenar** con un valor con `.fillna(valor)`. Útil cuando tiene sentido un valor por defecto (por ejemplo, `0` en un conteo).

> 🚫 **Nunca inventes un valor faltante.** Rellenar un monto desconocido con un número "parecido" falsea el análisis. Si no puedes recuperar el dato real, elimina la fila o márcala — pero no te inventes la cifra. En datos públicos, la honestidad del dato es parte del trabajo.


In [ ]:
df = cargar()

print("Antes  ->", len(df), "filas")
print(df["monto"].isnull().sum(), "fila(s) con monto faltante\n")

# Eliminamos las filas a las que les falta el monto
df_sin_faltantes = df.dropna(subset=["monto"])
print("Después ->", len(df_sin_faltantes), "filas")

## 5. Duplicados
Una **fila duplicada** es una fila idéntica a otra (mismo rubro y mismo monto). Suele pasar cuando un registro se carga dos veces. Si no la quitas, **inflas los totales**: sumarías ese monto dos veces.

- `.duplicated()` → marca con `True` las filas repetidas (deja la primera como `False`).
- `.drop_duplicates()` → elimina las repetidas y conserva una.

Con `keep="first"` (por defecto) conserva la primera aparición; con `keep="last"`, la última.


In [ ]:
df = cargar()

print("Antes  ->", len(df), "filas, duplicadas:", df.duplicated().sum())
df_sin_dup = df.drop_duplicates()
print("Después ->", len(df_sin_dup), "filas, duplicadas:", df_sin_dup.duplicated().sum())

## ⚠️ Errores típicos de principiante
- **Limpiar sin diagnosticar.** Si no corres `info()`, `isnull().sum()` y `duplicated().sum()` primero, arreglas a ciegas y se te escapan problemas. Diagnóstico siempre primero.
- **Olvidar reasignar.** `df["rubro"].str.strip()` **no modifica** la tabla: calcula un resultado y lo bota. Hay que guardar: `df["rubro"] = df["rubro"].str.strip()`.
- **Sumar texto.** Si `monto` es *object*, `df["monto"].sum()` pega los textos en vez de sumarlos. Convierte a número primero.
- **Confundir el punto.** Quitar mal los separadores de miles cambia el valor (`584.975` interpretado como decimal ≠ `584975`).
- **Buscar duplicados antes de normalizar el texto.** Dos filas "iguales" escritas con distinta capitalización **no** se detectan como duplicadas hasta que normalizas el texto.
- **Rellenar faltantes con cualquier cosa.** `fillna(0)` en una columna de montos baja los promedios y miente. Elige el método según lo que signifique el dato.


---
# 🛠️ Ejercicios
Cinco ejercicios sobre el *export crudo* de rubros de ChileCompra. Cada uno empieza con `df = cargar()` para partir de la tabla sucia original. Reemplaza cada `TODO`, ejecuta tu celda y luego la **celda de chequeo** (✅ / ❌).


## Ejercicio 01 — Diagnóstico
Antes de limpiar, mide el desorden. A partir de `df`, calcula:
- `n_faltantes`: cuántas celdas vacías hay en la columna `monto`.
- `n_duplicados`: cuántas filas están duplicadas en toda la tabla.

Pistas: `.isnull().sum()` sobre una columna; `.duplicated().sum()` sobre el DataFrame.

In [ ]:
df = cargar()

n_faltantes = df["monto"].isnull().sum()
n_duplicados = df.duplicated().sum()

print("Faltantes en monto:", n_faltantes)
print("Filas duplicadas:", n_duplicados)

In [ ]:
try:
    assert int(n_faltantes) == 1, f"Esperaba 1 faltante, obtuve {n_faltantes}"
    assert int(n_duplicados) == 1, f"Esperaba 1 duplicado, obtuve {n_duplicados}"
    print("✅ Correcto. Ejercicio 01 superado: diagnosticaste el dataset.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir n_faltantes o n_duplicados:", e)

## Ejercicio 02 — Limpiar el texto
Normaliza la columna `rubro`: quita los espacios sobrantes y deja cada rubro con la primera letra en mayúscula y el resto en minúscula.

Guarda el resultado **en la misma columna** `df["rubro"]`. Pista: encadena `.str.strip()` y `.str.capitalize()`.

In [ ]:
df = cargar()

df["rubro"] = df["rubro"].str.strip().str.capitalize()

for r in df["rubro"].unique():
    print(repr(r))

In [ ]:
try:
    valores = set(df["rubro"])
    assert "Medicamentos y productos farmacéuticos" in valores, "El rubro con espacios no quedó normalizado"
    assert "Equipamiento y suministros médicos" in valores, "El rubro en MAYÚSCULAS no quedó normalizado"
    assert not any(r != r.strip() for r in df["rubro"]), "Quedaron espacios sobrantes"
    print("✅ Correcto. Ejercicio 02 superado: texto normalizado.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta la columna rubro:", e)

## Ejercicio 03 — De texto a número
Escribe la función `a_entero(texto)` que recibe un monto como texto (por ejemplo `"$584.975.029.638"`) y devuelve el **número entero** correspondiente (`584975029638`).

Pasos: quitar el `$`, quitar los puntos, quitar espacios y convertir con `int(...)`.

In [ ]:
def a_entero(texto):
    limpio = str(texto).replace("$", "").replace(".", "").strip()
    return int(limpio)

print(a_entero("$584.975.029.638"))
print(a_entero("$1.000"))

In [ ]:
try:
    assert a_entero("$584.975.029.638") == 584975029638, "El monto grande no se convirtió bien"
    assert a_entero("$1.000") == 1000, "El monto pequeño no se convirtió bien"
    assert a_entero("$432.209.437.744") == 432209437744, "Revisa la conversión con otro monto"
    assert isinstance(a_entero("$1.000"), int), "Debe devolver un entero (int), no texto"
    print("✅ Correcto. Ejercicio 03 superado: montos convertidos a número.")
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir la función a_entero:", e)
except ValueError as e:
    print("❌ Revisa la conversión (¿quitaste $ y los puntos?):", e)

## Ejercicio 04 — El pipeline completo
Junta todo en una sola función `limpiar(df)` que reciba la tabla sucia y devuelva una **limpia y lista para analizar**. Debe, en este orden:

1. Trabajar sobre una copia (`df.copy()`), para no alterar el original.
2. Normalizar `rubro` (quitar espacios + capitalizar).
3. Eliminar las filas con `monto` faltante (`.dropna(subset=["monto"])`).
4. Eliminar filas duplicadas (`.drop_duplicates()`).
5. Convertir `monto` a entero usando tu función `a_entero` del Ejercicio 03.
6. Devolver la tabla limpia.

Reutiliza `a_entero`: aplícala a la columna con `.apply(a_entero)`.

In [ ]:
def limpiar(df):
    out = df.copy()
    out["rubro"] = out["rubro"].str.strip().str.capitalize()
    out = out.dropna(subset=["monto"])
    out = out.drop_duplicates()
    out["monto"] = out["monto"].apply(a_entero)
    return out

limpio = limpiar(cargar())
limpio

In [ ]:
try:
    assert limpio["monto"].isnull().sum() == 0, "Quedaron montos faltantes"
    assert limpio.duplicated().sum() == 0, "Quedaron filas duplicadas"
    assert pd.api.types.is_integer_dtype(limpio["monto"]), "La columna monto no quedó numérica entera"
    assert len(limpio) == 5, f"Esperaba 5 rubros limpios, obtuve {len(limpio)}"
    fila = limpio[limpio["rubro"] == "Medicamentos y productos farmacéuticos"]
    assert int(fila["monto"].iloc[0]) == 584975029638, "El monto de Medicamentos no quedó bien"
    print("✅ Correcto. Ejercicio 04 superado: ¡pipeline de limpieza completo!")
    print("Gasto total (5 rubros, datos limpios): ${:,.0f} CLP".format(limpio["monto"].sum()).replace(",", "."))
except AssertionError as e:
    print("❌ Aún no:", e)
except NameError as e:
    print("❌ Falta definir limpiar (o a_entero del Ej. 03):", e)

## Ejercicio 05 — Media vs mediana: ¿qué resume mejor el gasto?

Ya tienes la tabla **limpia** (5 rubros, montos como número entero). Ahora vas a **interpretar**, no solo calcular.

Cuando alguien te pregunta "¿cuánto gasta *en promedio* el Estado por rubro?", la respuesta no es tan inocente. La **media** (promedio) se deja arrastrar por los valores extremos; la **mediana** (el valor del medio) no. Comparar ambas te dice si la distribución está **inclinada** hacia un lado.

Sobre la tabla limpia (`limpio`), calcula:
- `media`: el promedio de la columna `monto` (`.mean()`).
- `mediana`: el valor central de la columna `monto` (`.median()`).

Luego **observa cuál es mayor** y elige la conclusión correcta asignando la letra a `conclusion` (por ejemplo `conclusion = "B"`):

- **A.** La media es **mayor** que la mediana: hay uno o pocos rubros con montos muy altos que **inflan** el promedio. El "promedio" exagera el gasto típico.
- **B.** La media es **menor** que la mediana: el rubro de monto más bajo **arrastra el promedio hacia abajo**, así que la media **subestima** el gasto del rubro típico. La mediana refleja mejor el caso central.
- **C.** La media y la mediana son **exactamente iguales**: el gasto está perfectamente repartido entre los rubros y da lo mismo cuál uses.

> Pista: compara los dos números que imprimiste. La conclusión correcta se *sigue* de cuál resultó mayor — no la adivines.

*(Opcional, no se corrige)* En `reflexion`, escribe en una frase qué número reportarías a una autoridad si te pregunta "el gasto típico por rubro" y por qué.

In [ ]:
limpio = limpiar(cargar())   # tabla ya limpia del Ejercicio 04

media = limpio["monto"].mean()
mediana = limpio["monto"].median()

# La media (482,6 mil millones) resulta MENOR que la mediana (500,6 mil millones):
# el rubro de monto mas bajo arrastra el promedio hacia abajo -> opcion B.
conclusion = "B"

reflexion = "Reportaria la mediana, porque la media subestima el gasto del rubro tipico al ser arrastrada hacia abajo por el rubro mas barato."

print("Media  :", media)
print("Mediana:", mediana)
print("Conclusion elegida:", conclusion)

In [ ]:
try:
    # Recomputamos los valores esperados desde el CSV (sin hardcodear cifras)
    _df = cargar()
    _df["rubro"] = _df["rubro"].str.strip().str.capitalize()
    _df = _df.dropna(subset=["monto"]).drop_duplicates()
    _df["monto"] = _df["monto"].str.replace("$", "", regex=False).str.replace(".", "", regex=False).astype(int)
    _media_esp = _df["monto"].mean()
    _mediana_esp = _df["monto"].median()
    # La conclusion correcta se deduce de la comparacion recomputada
    if _media_esp > _mediana_esp:
        _letra_esp = "A"
    elif _media_esp < _mediana_esp:
        _letra_esp = "B"
    else:
        _letra_esp = "C"

    assert media is not None and mediana is not None, "Aun no calculaste media y/o mediana"
    assert abs(float(media) - _media_esp) < 1, f"La media no coincide. Esperaba {_media_esp:,.0f}"
    assert abs(float(mediana) - _mediana_esp) < 1, f"La mediana no coincide. Esperaba {_mediana_esp:,.0f}"
    assert conclusion is not None, "Aun no elegiste una conclusion (asigna A, B o C)"
    assert str(conclusion).strip().upper() == _letra_esp, "Esa no es la conclusion que se sigue de comparar media y mediana. Mira de nuevo cual numero resulto mayor."
    print("\u2705 Correcto. Ejercicio 05 superado: comparaste media vs mediana e interpretaste el sesgo.")
    print(f"   Media = ${_media_esp:,.0f} < Mediana = ${_mediana_esp:,.0f}  ->  la media subestima el gasto tipico (opcion B).".replace(",", "."))
except AssertionError as e:
    print("\u274c Aun no:", e)
except NameError as e:
    print("\u274c Falta definir media, mediana o conclusion (corriste el Ejercicio 04 con limpiar y a_entero?):", e)


---
## ¿Terminaste?
Si las cinco celdas de chequeo muestran ✅, completaste **M4**. 🎉

Aprendiste el flujo de limpieza que usa todo analista: **diagnosticar** (faltantes, duplicados, tipos), **normalizar texto**, **convertir tipos**, **resolver faltantes** y **quitar duplicados** — y a empaquetarlo en una función reutilizable. Sobre todo, viste por qué los datos públicos llegan sucios (son autodeclarados) y por qué limpiarlos bien es un acto de rigor.

En **M5 · SQL fundamentos** darás un giro: en vez de traer toda la tabla a Python para filtrarla, le pedirás directamente a una **base de datos** justo lo que necesitas, con el lenguaje SQL. Es la herramienta que comparten las dos líneas del bootcamp (Analytics y Data Science) y la que más vas a usar para consultar datos del Estado.